# Rang Conveyor Belt Classifier
**AIC-401 Deep Learning — Assignment 01**

This notebook trains a CNN from scratch to classify plastic objects by color:
- 🔵 Blue → Conveyor Belt A
- 🟡 Yellow → Conveyor Belt B
- 🟣 Purple → Conveyor Belt C

## Cell 1 — Install Dependencies
Run this cell only once (especially on Google Colab)

## Cell 2 — Import Libraries

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.21.0
GPU Available: True


## Cell 3 — Configuration
Set all hyperparameters and paths here. Change `DATASET_PATH` to your actual dataset folder.

In [ ]:
# ─── PATHS ───────────────────────────────────────────────────────────────────
DATASET_PATH   = "dataset"          # Root folder containing blue/yellow/purple subfolders
MODEL_SAVE_PATH = "saved_model/conveyor_model.h5"
TFLITE_SAVE_PATH = "saved_model/conveyor_model.tflite"

# ─── IMAGE SETTINGS ──────────────────────────────────────────────────────────
IMG_SIZE    = (128, 128)            # Resize all images to 128x128
BATCH_SIZE  = 32

# ─── TRAINING SETTINGS ───────────────────────────────────────────────────────
EPOCHS      = 30
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2              # 80% train, 20% validation

# ─── CLASSES ─────────────────────────────────────────────────────────────────
CLASS_NAMES = ['blue', 'purple', 'yellow']   # alphabetical order (Keras default)
CLASS_LABELS = {
    'blue':   'Conveyor Belt A',
    'yellow': 'Conveyor Belt B',
    'purple': 'Conveyor Belt C'
}

# Create output directory
os.makedirs("saved_model", exist_ok=True)

print("Configuration set!")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

FileNotFoundError: [Errno 2] No such file or directory: 'saved_model'

## Cell 4 — Mount Google Drive (Only on Colab)
Skip this cell if running locally.

In [ ]:
# ── Uncomment below if using Google Colab ──

# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = "/content/drive/MyDrive/khilona-classifier/dataset"

print("(Skipped — running locally)")

## Cell 5 — Verify Dataset Structure
Make sure your dataset folder looks like:
```
dataset/
  blue/     ← your blue object photos
  yellow/   ← your yellow object photos
  purple/   ← your purple object photos
```

In [ ]:
# Verify dataset structure and count images
print(f"Dataset path: {os.path.abspath(DATASET_PATH)}\n")

total_images = 0
for class_name in sorted(os.listdir(DATASET_PATH)):
    class_folder = os.path.join(DATASET_PATH, class_name)
    if os.path.isdir(class_folder):
        count = len([
            f for f in os.listdir(class_folder)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        total_images += count
        belt = CLASS_LABELS.get(class_name, '?')
        print(f"  📁 {class_name:10s} → {belt:20s} | {count} images")

print(f"\n  Total images: {total_images}")

## Cell 6 — Preview Sample Images

In [ ]:
from tensorflow.keras.preprocessing.image import load_img

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Sample Images from Dataset', fontsize=16, fontweight='bold')

for row, class_name in enumerate(sorted(os.listdir(DATASET_PATH))):
    class_folder = os.path.join(DATASET_PATH, class_name)
    if not os.path.isdir(class_folder):
        continue

    images = [
        f for f in os.listdir(class_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ][:4]  # show 4 samples per class

    for col, img_name in enumerate(images):
        img = load_img(os.path.join(class_folder, img_name), target_size=IMG_SIZE)
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{class_name}\n{CLASS_LABELS.get(class_name, '')}", fontsize=8)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('saved_model/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sample images saved.")

## Cell 7 — Data Augmentation & Loading
Augmentation artificially increases dataset variety by flipping, rotating, zooming etc. This helps the model generalize better.

In [ ]:
# Training data generator — WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255,              # Normalize pixel values to [0, 1]
    validation_split=VALIDATION_SPLIT,
    rotation_range=20,            # Randomly rotate up to 20 degrees
    width_shift_range=0.1,        # Randomly shift horizontally
    height_shift_range=0.1,       # Randomly shift vertically
    shear_range=0.1,              # Shear transformation
    zoom_range=0.15,              # Random zoom
    horizontal_flip=True,         # Randomly flip horizontally
    brightness_range=[0.8, 1.2],  # Random brightness change
    fill_mode='nearest'
)

# Validation generator — NO augmentation, only rescaling
val_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=VALIDATION_SPLIT
)

# Load training set
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',     # One-hot encoding for 3 classes
    subset='training',
    shuffle=True,
    seed=42
)

# Load validation set
val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print(f"\nClass indices: {train_generator.class_indices}")
print(f"Training samples:   {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

In [ ]:
# ── Count Train & Validation Images Per Class ─────────────────────────────────

class_indices = train_generator.class_indices
idx_to_class  = {v: k for k, v in class_indices.items()}

train_counts = {}
val_counts   = {}

# Count training images per class
for idx, label in zip(train_generator.classes, train_generator.filenames):
    cls = idx_to_class[idx]
    train_counts[cls] = train_counts.get(cls, 0) + 1

# Count validation images per class
for idx, label in zip(val_generator.classes, val_generator.filenames):
    cls = idx_to_class[idx]
    val_counts[cls] = val_counts.get(cls, 0) + 1

# ── Print Summary Table ────────────────────────────────────────────────────────
print(f"{'='*55}")
print(f"  {'Class':<12} {'Train':>8} {'Validation':>12} {'Total':>8}")
print(f"{'─'*55}")

for cls in sorted(class_indices.keys()):
    tr  = train_counts.get(cls, 0)
    val = val_counts.get(cls, 0)
    tot = tr + val
    print(f"  {cls:<12} {tr:>8} {val:>12} {tot:>8}")

print(f"{'─'*55}")
total_tr  = sum(train_counts.values())
total_val = sum(val_counts.values())
total_all = total_tr + total_val
print(f"  {'TOTAL':<12} {total_tr:>8} {total_val:>12} {total_all:>8}")
print(f"{'='*55}")
print(f"\n  Split ratio — Train: {total_tr/total_all*100:.1f}%  |  Val: {total_val/total_all*100:.1f}%")

## Cell 8 — Build the CNN Model
A 3-block CNN with BatchNormalization and Dropout — trained from scratch, no pretrained weights.

In [ ]:
def build_model(input_shape=(128, 128, 3), num_classes=3):
    """
    Custom CNN Architecture:
    - 3 Convolutional blocks (Conv → BN → ReLU → MaxPool)
    - GlobalAveragePooling (better than Flatten for mobile deployment)
    - Dense layers with Dropout for regularization
    - Softmax output for 3-class classification
    """
    model = keras.Sequential([

        # ── Block 1 ──────────────────────────────────────────────
        # 32 filters, 3x3 kernel — learns basic color/edge features
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),      # Stabilizes and speeds up training
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),      # Reduces spatial size: 128→64
        layers.Dropout(0.25),             # Prevents overfitting

        # ── Block 2 ──────────────────────────────────────────────
        # 64 filters — learns more complex color patterns
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),      # 64→32
        layers.Dropout(0.25),

        # ── Block 3 ──────────────────────────────────────────────
        # 128 filters — learns high-level color representations
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),      # 32→16
        layers.Dropout(0.25),

        # ── Classifier Head ───────────────────────────────────────
        # GlobalAveragePooling: averages each feature map → compact vector
        # Better than Flatten for TFLite mobile deployment
        layers.GlobalAveragePooling2D(),

        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),

        # Output: 3 neurons with softmax → probability for each class
        layers.Dense(num_classes, activation='softmax')
    ])
    return model


model = build_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=3)
model.summary()

## Cell 9 — Compile the Model

In [ ]:
model.compile(
    # Adam optimizer: adaptive learning rate, works well for CNNs
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),

    # Categorical crossentropy: standard loss for multi-class classification
    loss='categorical_crossentropy',

    # Track accuracy during training
    metrics=['accuracy']
)

print(f"Total parameters: {model.count_params():,}")
print("Model compiled successfully!")

## Cell 10 — Set Up Training Callbacks
Callbacks automatically improve training: save best model, reduce LR when stuck, stop early if needed.

In [ ]:
callbacks = [

    # Save the best model (by validation accuracy) automatically
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),

    # Reduce learning rate by 50% if val_accuracy doesn't improve for 5 epochs
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    # Stop training early if val_accuracy doesn't improve for 10 epochs
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

print("Callbacks configured:")
print("  ✓ ModelCheckpoint — saves best model")
print("  ✓ ReduceLROnPlateau — lowers LR when stuck")
print("  ✓ EarlyStopping — stops if no improvement")

## Cell 11 — Train the Model 🚀

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# Then in model.fit():
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight_dict,   # ← add this line
    verbose=1
)

## Cell 12 — Plot Training Curves
Visualize accuracy and loss over epochs. Take a screenshot of this for your assignment report.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=16, fontweight='bold')

epochs_ran = range(1, len(history.history['accuracy']) + 1)

# ── Accuracy Plot ─────────────────────────────────────────────────────────────
ax1.plot(epochs_ran, history.history['accuracy'],     'b-o', label='Train Accuracy', markersize=3)
ax1.plot(epochs_ran, history.history['val_accuracy'], 'r-o', label='Val Accuracy',   markersize=3)
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# ── Loss Plot ─────────────────────────────────────────────────────────────────
ax2.plot(epochs_ran, history.history['loss'],     'b-o', label='Train Loss', markersize=3)
ax2.plot(epochs_ran, history.history['val_loss'], 'r-o', label='Val Loss',   markersize=3)
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('saved_model/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to saved_model/training_curves.png")

## Cell 13 — Evaluate on Validation Set
Get final accuracy and detailed per-class metrics.

In [ ]:
# Load the best saved model
best_model = keras.models.load_model(MODEL_SAVE_PATH)

# Evaluate on validation set
val_loss, val_accuracy = best_model.evaluate(val_generator, verbose=0)
print(f"{'='*40}")
print(f"  Validation Loss:     {val_loss:.4f}")
print(f"  Validation Accuracy: {val_accuracy*100:.2f}%")
print(f"{'='*40}")

# Get per-class predictions
val_generator.reset()
predictions = best_model.predict(val_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = val_generator.classes
class_names = list(val_generator.class_indices.keys())

# Detailed classification report
print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes, target_names=class_names))

## Cell 14 — Confusion Matrix
Shows exactly which classes are being confused. Take a screenshot for your report.

In [ ]:
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=[f"{c}\n({CLASS_LABELS.get(c,'')})" for c in class_names],
    yticklabels=[f"{c}\n({CLASS_LABELS.get(c,'')})" for c in class_names]
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('Actual Class', fontsize=12)
plt.xlabel('Predicted Class', fontsize=12)
plt.tight_layout()
plt.savefig('saved_model/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to saved_model/confusion_matrix.png")

## Cell 15 — Test on a Single Image
Simulate what happens when the evaluator shows a new object.

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def predict_conveyor_belt(image_path, model, class_indices):
    """
    Predict which conveyor belt an object should go to.
    Args:
        image_path: path to the image file
        model: trained Keras model
        class_indices: dict mapping class names to indices
    Returns:
        Predicted class and belt assignment
    """
    # Reverse the class_indices dict: {0: 'blue', 1: 'purple', 2: 'yellow'}
    idx_to_class = {v: k for k, v in class_indices.items()}

    # Load and preprocess image
    img = load_img(image_path, target_size=IMG_SIZE)
    img_array = img_to_array(img) / 255.0          # Normalize
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Predict
    predictions = model.predict(img_array, verbose=0)[0]
    predicted_idx = np.argmax(predictions)
    predicted_class = idx_to_class[predicted_idx]
    confidence = predictions[predicted_idx] * 100

    # Display result
    belt = CLASS_LABELS[predicted_class]
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Color: {predicted_class.upper()}\n→ {belt}\nConfidence: {confidence:.1f}%",
              fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return predicted_class, belt, confidence


# ── TEST: Change this to any image path ──────────────────────────────────────
TEST_IMAGE = "dataset/blue/your_image.jpg"    # ← change this

if os.path.exists(TEST_IMAGE):
    color, belt, conf = predict_conveyor_belt(TEST_IMAGE, best_model, train_generator.class_indices)
    print(f"Result: {color.upper()} object → {belt} ({conf:.1f}% confidence)")
else:
    print(f"Image not found: {TEST_IMAGE}")
    print("Update TEST_IMAGE path to any image from your dataset to test.")

## Cell 16 — Export to TFLite (for Flutter App)
Converts the Keras model to a lightweight format that runs on Android.

In [ ]:
print("Converting model to TFLite format...")

# Load best model
best_model = keras.models.load_model(MODEL_SAVE_PATH)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)

# Optional: Optimize for mobile (reduces size, minimal accuracy loss)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save the TFLite model
with open(TFLITE_SAVE_PATH, 'wb') as f:
    f.write(tflite_model)

# Report file sizes
h5_size     = os.path.getsize(MODEL_SAVE_PATH) / 1024
tflite_size = os.path.getsize(TFLITE_SAVE_PATH) / 1024

print(f"\n✅ TFLite model saved to: {TFLITE_SAVE_PATH}")
print(f"   Keras model size:  {h5_size:.1f} KB")
print(f"   TFLite model size: {tflite_size:.1f} KB")
print(f"   Size reduction:    {(1 - tflite_size/h5_size)*100:.1f}%")
print(f"\n➡️  Copy this file to: app/assets/conveyor_model.tflite")

## Cell 17 — Summary of All Saved Files

In [ ]:
print("="*50)
print("  KHILONA CLASSIFIER — TRAINING COMPLETE")
print("="*50)
print(f"\n  Final Validation Accuracy: {val_accuracy*100:.2f}%")
print(f"  Final Validation Loss:     {val_loss:.4f}")
print(f"\n  Saved Files:")

for fname in os.listdir('saved_model'):
    fpath = os.path.join('saved_model', fname)
    size = os.path.getsize(fpath) / 1024
    print(f"    📄 {fname:35s} {size:8.1f} KB")

print(f"\n  Class Mapping:")
for class_name, idx in sorted(train_generator.class_indices.items(), key=lambda x: x[1]):
    print(f"    Index {idx} → {class_name:10s} → {CLASS_LABELS.get(class_name, '?')}")

print("\n  Next step: Copy conveyor_model.tflite to app/assets/")
print("="*50)